# 02c – Test-time augmentation and ensemble (no retraining)

Can the trained round-3 models (notebook 02d) predict better **without training again**?

- **Test-time augmentation (TTA):** every patch is also predicted flipped and rotated; the predictions are turned back
  and averaged. Land cover has no "up" direction, so all versions are equally valid views of the same ground.
  - `flip4`: original, left-right flip, up-down flip, both (= 180° rotation)
  - `dihedral8`: all 4 rotations, each with and without a flip
- **Ensemble:** the probabilities of all candidate models (SegFormer-B3, SegFormer-B2, DeepLabV3+, ResNet50 U-Net) are averaged.

**Rule:** every setting is compared on the **validation** set only. The best setting is chosen there,
and only that setting (plus the model selected in 02d as reference) is evaluated on the test set, once.

As a check, the single models without TTA must give the same validation mIoU as in notebook 02d.

# Preliminaries

In [1]:
import sys
import subprocess
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").exists()

# keras-hub (for SegFormer) is installed WITHOUT its dependencies: a normal install
# pulls in another TensorFlow version and breaks the existing one.
# The keras-hub version must fit the installed Keras: 0.33.0 needs Keras >= 3.15,
# 0.24.0 works with Keras 3.8 - 3.14 (tested with TensorFlow 2.20 + Keras 3.11.2).
from importlib.metadata import version as package_version, PackageNotFoundError

keras_version = tuple(int(part) for part in package_version("keras").split(".")[:2])
KERAS_HUB = "0.33.0" if keras_version >= (3, 15) else "0.24.0"

try:
    installed = package_version("keras-hub")
except PackageNotFoundError:
    installed = None

if installed != KERAS_HUB:
    print(f"installing keras-hub {KERAS_HUB} (found: {installed}) for Keras {package_version('keras')}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall",
                    f"keras-hub=={KERAS_HUB}"], check=True)
    if "keras_hub" in sys.modules:
        raise RuntimeError("keras-hub was replaced - restart the kernel and run this cell again")

# small helpers keras-hub needs: downloads from Kaggle Models / Hugging Face (no TensorFlow dependency)
missing = []
for package in ["kagglehub", "huggingface_hub", "regex", "rich"]:
    try:
        package_version(package.replace("_", "-"))
    except PackageNotFoundError:
        missing.append(package)
if missing:
    print("installing", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import keras_hub
print("keras-hub:", keras_hub.__version__)

keras-hub: 0.24.0


In [2]:
import os
import json
import time

import numpy as np
import tensorflow as tf
import keras

GPUS = tf.config.list_physical_devices("GPU")
if GPUS:
    keras.mixed_precision.set_global_policy("mixed_float16")
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__, "| GPUs:", GPUS)

model_folder = Path("models/round3")
patch_folder = Path("data/patches_v2")          # validation and test come from the round-1 set only

class_info = json.loads((patch_folder / "classes.json").read_text())
assert class_info.get("plan_version") == "v2", "data/patches_v2 must be the round-1 set (made by notebook 01c)"
CLASSES = class_info["classes"]
IGNORE = class_info["ignore"]
N_CLASSES = len(CLASSES)

norm = json.loads((model_folder / "normalisation.json").read_text())
data = {split: {"x": np.load(patch_folder / f"{split}_x.npy"), "y": np.load(patch_folder / f"{split}_y.npy")}
        for split in ["validation", "test"]}
for split in data:
    print(split, data[split]["x"].shape)

TensorFlow: 2.20.0 | Keras: 3.11.2 | GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
validation (864, 256, 256, 2)
test (864, 256, 256, 2)


## Same normalisation and metrics as in training

The normalisation values are read from `models/round3/normalisation.json` (computed on the training data in 02d;
all candidate models were trained with these values).

In [3]:
def normalise(x):
    x = np.clip(x.astype(np.float32), np.float32(norm["clip_low"]), np.float32(norm["clip_high"]))
    if norm["add_ratio"]:
        x = np.concatenate([x, x[..., 0:1] - x[..., 1:2]], axis=-1)
    return (x - np.float32(norm["mean"])) / np.float32(norm["std"])


def confusion(y_true, y_pred):
    valid = y_true != IGNORE
    index = N_CLASSES * y_true[valid].astype(np.int64) + y_pred[valid]
    return np.bincount(index, minlength=N_CLASSES ** 2).reshape(N_CLASSES, N_CLASSES)


def scores(cm):
    cm = np.asarray(cm, dtype=np.float64)
    tp = np.diag(cm)
    actual = cm.sum(axis=1)
    predicted = cm.sum(axis=0)
    union = actual + predicted - tp
    iou = np.divide(tp, union, out=np.full(tp.shape, np.nan), where=union > 0)
    f1 = np.divide(2 * tp, actual + predicted, out=np.full(tp.shape, np.nan), where=(actual + predicted) > 0)
    return {"oa": tp.sum() / cm.sum(), "miou": np.nanmean(iou), "iou": iou, "f1": f1}

## Flips and rotations

Each transform has an inverse, so the prediction can be turned back onto the original pixel grid before averaging.

In [4]:
def dihedral(k, flip):
    # rotate by k * 90 degrees, optionally flip left-right first; arrays are (batch, height, width, channels)
    def forward(a):
        a = a[:, :, ::-1] if flip else a
        return np.rot90(a, k, axes=(1, 2))

    def inverse(a):
        a = np.rot90(a, -k, axes=(1, 2))
        return a[:, :, ::-1] if flip else a

    return forward, inverse


TRANSFORMS = {(k, flip): dihedral(k, flip) for k in range(4) for flip in (False, True)}
TTA_SETS = {
    "none": [(0, False)],
    "flip4": [(0, False), (0, True), (2, True), (2, False)],     # original, left-right, up-down, 180 degrees
    "dihedral8": list(TRANSFORMS),
}

# check on random data: every transform followed by its inverse gives back the input
test_array = np.random.rand(2, 8, 8, 3)
for key, (forward, inverse) in TRANSFORMS.items():
    assert np.array_equal(inverse(forward(test_array)), test_array), key
print("all 8 transforms are exactly invertible")

all 8 transforms are exactly invertible


## Models

In [5]:
CANDIDATES = ["segformer_b3", "segformer_b2", "deeplabv3plus_resnet50", "unet_resnet50"]
SELECTED = json.loads((model_folder / "model_info.json").read_text())["model"]     # the model chosen in 02d
print("selected in 02d:", SELECTED)

models = {}
for name in CANDIDATES:
    path = model_folder / name / "best.keras"
    if path.exists():
        models[name] = keras.models.load_model(path, compile=False)
        print("loaded", name)
    else:
        print("missing", path)
assert SELECTED in models, "the model selected in 02d is needed"

settings = [(name, tta) for name in models for tta in TTA_SETS]
if len(models) > 1:
    settings += [("ensemble", tta) for tta in TTA_SETS]
print(len(settings), "settings to compare")

selected in 02d: segformer_b3


loaded segformer_b3
loaded segformer_b2
loaded deeplabv3plus_resnet50
loaded unet_resnet50
15 settings to compare


## Evaluation

For every batch, each model predicts all 8 transformed versions once. The settings are averages of these predictions,
so nothing is predicted twice. Only confusion matrices are kept, not the probabilities (they would need several GB).

In [6]:
BATCH = 16


def evaluate(split, settings):
    x_all, y_all = data[split]["x"], data[split]["y"]
    matrices = {s: np.zeros((N_CLASSES, N_CLASSES), dtype=np.int64) for s in settings}
    needed = {name for name, _ in settings if name != "ensemble"} | (set(models) if any(n == "ensemble" for n, _ in settings) else set())
    needed_transforms = sorted({t for _, tta in settings for t in TTA_SETS[tta]})
    seconds = {name: 0.0 for name in needed}

    for start in range(0, len(x_all), BATCH):
        x = normalise(x_all[start:start + BATCH])
        y = y_all[start:start + BATCH]

        # probabilities[name][transform] on the original grid
        probabilities = {}
        for name in needed:
            begin = time.time()
            probabilities[name] = {}
            for t in needed_transforms:
                forward, inverse = TRANSFORMS[t]
                p = np.asarray(models[name].predict_on_batch(np.ascontiguousarray(forward(x))), dtype=np.float32)
                probabilities[name][t] = inverse(p)
            seconds[name] += time.time() - begin

        for name, tta in settings:
            members = list(models) if name == "ensemble" else [name]
            mean = sum(probabilities[m][t] for m in members for t in TTA_SETS[tta])
            matrices[(name, tta)] += confusion(y, np.argmax(mean, axis=-1).astype(np.uint8))
    return matrices, seconds


start = time.time()
valid_matrices, _ = evaluate("validation", settings)
print(f"validation done in {(time.time() - start) / 60:.1f} min")

validation done in 2.4 min


In [7]:
valid_results = {s: scores(cm) for s, cm in valid_matrices.items()}
reference = valid_results[(SELECTED, "none")]["miou"]

print(f"{'setting':<34}{'val OA':>8}{'val mIoU':>10}{'vs selected':>13}")
for (name, tta), r in sorted(valid_results.items(), key=lambda item: -item[1]["miou"]):
    print(f"{name + ' + ' + tta:<34}{r['oa']:8.3f}{r['miou']:10.3f}{r['miou'] - reference:+13.3f}")

print("\nCheck against notebook 02d (best validation mIoU during training, no TTA):")
for name in models:
    history = json.loads((model_folder / name / "history.json").read_text())
    print(f"  {name:<24} 02d: {max(history['val_miou']):.3f}   here: {valid_results[(name, 'none')]['miou']:.3f}")

setting                             val OA  val mIoU  vs selected
ensemble + dihedral8                 0.642     0.379       +0.015
ensemble + flip4                     0.641     0.378       +0.014
ensemble + none                      0.640     0.377       +0.013
segformer_b3 + dihedral8             0.627     0.367       +0.002
segformer_b3 + flip4                 0.627     0.366       +0.002
segformer_b3 + none                  0.625     0.364       +0.000
segformer_b2 + dihedral8             0.624     0.364       -0.000
segformer_b2 + flip4                 0.623     0.363       -0.001
deeplabv3plus_resnet50 + dihedral8   0.628     0.362       -0.002
segformer_b2 + none                  0.620     0.361       -0.003
unet_resnet50 + dihedral8            0.620     0.361       -0.003
deeplabv3plus_resnet50 + flip4       0.625     0.360       -0.005
unet_resnet50 + flip4                0.618     0.358       -0.006
deeplabv3plus_resnet50 + none        0.618     0.353       -0.011
unet_resne

## Choice on the validation set

A setting is only worth its extra prediction time if it is clearly better: at least +0.005 validation mIoU
over the model selected in 02d. Otherwise that model stays as it is.

In [8]:
MIN_GAIN = 0.005

best_setting = max(valid_results, key=lambda s: valid_results[s]["miou"])
if valid_results[best_setting]["miou"] - reference < MIN_GAIN:
    best_setting = (SELECTED, "none")
name, tta = best_setting
members = list(models) if name == "ensemble" else [name]
cost = len(members) * len(TTA_SETS[tta])
print(f"chosen on validation: {name} + {tta}  ({cost} predictions per patch instead of 1)")

chosen on validation: ensemble + dihedral8  (32 predictions per patch instead of 1)


# Test set (chosen setting and the model selected in 02d as reference)

In [9]:
test_settings = sorted({best_setting, (SELECTED, "none")})
test_matrices, seconds = evaluate("test", test_settings)
test_results = {s: scores(cm) for s, cm in test_matrices.items()}

for s in test_settings:
    r = test_results[s]
    print(f"{s[0] + ' + ' + s[1]:<34} test OA {r['oa']:.3f}   test mIoU {r['miou']:.3f}")

print(f"\n{'class':<26}" + "".join(f"{s[0][:12] + '+' + s[1]:>22}" for s in test_settings))
for k, class_name in enumerate(CLASSES):
    print(f"{class_name:<26}" + "".join(f"{test_results[s]['iou'][k]:22.3f}" for s in test_settings))

patches = len(data["test"]["x"])
for model_name, total in seconds.items():
    print(f"{model_name}: {1000 * total / patches:.1f} ms per patch for all transforms used here")

ensemble + dihedral8               test OA 0.699   test mIoU 0.432
segformer_b3 + none                test OA 0.692   test mIoU 0.428

class                         ensemble+dihedral8     segformer_b3+none
Trees (incl. mangroves)                    0.477                 0.480
Grassland / shrubland                      0.437                 0.413
Cropland                                   0.588                 0.581
Built-up                                   0.626                 0.613
Bare / sparse vegetation                   0.077                 0.079
Permanent water                            0.798                 0.793
Herbaceous wetland                         0.024                 0.038
deeplabv3plus_resnet50: 25.0 ms per patch for all transforms used here
unet_resnet50: 16.6 ms per patch for all transforms used here
segformer_b2: 10.7 ms per patch for all transforms used here
segformer_b3: 13.4 ms per patch for all transforms used here


## Save the decision for the prototype

In [10]:
decision = {
    "models": members,
    "tta": tta,
    "transforms": [list(t) for t in TTA_SETS[tta]],
    "predictions_per_patch": cost,
    "validation_miou": float(valid_results[best_setting]["miou"]),
    "selected_in_02d": SELECTED,
    "validation_miou_selected_alone": float(reference),
    "test_miou": float(test_results[best_setting]["miou"]),
    "test_oa": float(test_results[best_setting]["oa"]),
    "test_miou_selected_alone": float(test_results[(SELECTED, "none")]["miou"]),
}
(model_folder / "inference_choice.json").write_text(json.dumps(decision, indent=2))
print(json.dumps(decision, indent=2))

{
  "models": [
    "segformer_b3",
    "segformer_b2",
    "deeplabv3plus_resnet50",
    "unet_resnet50"
  ],
  "tta": "dihedral8",
  "transforms": [
    [
      0,
      false
    ],
    [
      0,
      true
    ],
    [
      1,
      false
    ],
    [
      1,
      true
    ],
    [
      2,
      false
    ],
    [
      2,
      true
    ],
    [
      3,
      false
    ],
    [
      3,
      true
    ]
  ],
  "predictions_per_patch": 32,
  "validation_miou": 0.37888979933612604,
  "selected_in_02d": "segformer_b3",
  "validation_miou_selected_alone": 0.36414766819013533,
  "test_miou": 0.43229530301855307,
  "test_oa": 0.6989329479358815,
  "test_miou_selected_alone": 0.4282085850053893
}
